# 🎨 Photo Colorization with cGAN

This notebook trains a conditional GAN (cGAN) to colorize grayscale photos.

**Architecture overview:**
- **Color space:** Lab — the network takes the L (lightness) channel as input and predicts the ab (color) channels
- **Generator:** U-Net with skip connections for high-fidelity spatial detail
- **Discriminator:** PatchGAN — classifies overlapping image patches as real/fake instead of the whole image
- **Loss:** Adversarial loss + L1 reconstruction loss (weighted by λ=100)

Runtime: **GPU (T4 or better recommended)**. Enable via *Runtime → Change runtime type → GPU*.

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Not available'}")
if torch.cuda.is_available():
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/colorize-ai'
import os
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/logs', exist_ok=True)

In [ ]:
!git clone -b claude/upbeat-bardeen-oro25c https://github.com/nadolskyidenisfpm21/colorize-ai /content/colorize-ai
%cd /content/colorize-ai
!pip install -q -r requirements.txt
import os
# Symlink to Drive for persistence
if not os.path.exists('checkpoints'):
    os.symlink(f'{DRIVE_DIR}/checkpoints', 'checkpoints')
if not os.path.exists('logs'):
    os.symlink(f'{DRIVE_DIR}/logs', 'logs')

## Dataset

Choose one of the three options below and run the corresponding cell.

### Option 1 — ImageNet-mini via Kaggle (recommended)
~1 000 classes, ~130k images. Requires a [Kaggle account](https://www.kaggle.com/) and API token (`kaggle.json`).
Good balance of diversity and download size (~3 GB).

### Option 2 — COCO 2017
~118k images across 80 categories (~18 GB). No Kaggle needed, but the download takes longer.
Great if you want a larger, more varied dataset.

### Option 3 — STL-10 (no sign-up needed, quickest)
5 000 unlabeled 96×96 images from 10 classes. Downloads automatically via torchvision.
Ideal for a quick smoke-test run before committing to a longer training job.

In [ ]:
# Option 1: Kaggle (recommended)
# Upload your kaggle.json first:
# from google.colab import files; files.upload()

# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d ifigotin/imagenetmini-1000 -p /content/data --unzip
# DATA_DIR = '/content/data/imagenet-mini/train'

# Option 2: COCO 2017 (118k images, ~18GB)
# !wget -q http://images.cocodataset.org/zips/train2017.zip -O /content/coco.zip
# !unzip -q /content/coco.zip -d /content/data/
# DATA_DIR = '/content/data/train2017'

# Option 3: Quick test with STL10 (5k images, no Kaggle needed)
import torchvision
import os
stl = torchvision.datasets.STL10(root='/content/data', split='unlabeled', download=True)
DATA_DIR = '/content/data/stl10_images'
os.makedirs(DATA_DIR, exist_ok=True)
# Save STL10 images as files for the dataset loader
from PIL import Image
import numpy as np
print("Saving STL10 images...")
for i, (img, _) in enumerate(stl):
    img.save(f'{DATA_DIR}/{i:05d}.png')
    if i % 1000 == 0: print(f'{i}/{len(stl)}')
print(f"Done. {len(stl)} images saved to {DATA_DIR}")

In [ ]:
EPOCHS = 100
BATCH_SIZE = 16  # reduce to 8 if OOM
IMAGE_SIZE = 256
LR = 2e-4
LAMBDA_L1 = 100

In [ ]:
!python train.py \
    --data_dir {DATA_DIR} \
    --epochs {EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --image_size {IMAGE_SIZE} \
    --lr {LR} \
    --lambda_l1 {LAMBDA_L1}

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs

In [ ]:
import torch, sys
sys.path.insert(0, '/content/colorize-ai')
from models.generator import UNetGenerator
from utils import load_checkpoint, lab_to_rgb
from dataset import ColorizationDataset
from config import cfg
import matplotlib.pyplot as plt
import numpy as np

device = 'cuda' if torch.cuda.is_available() else 'cpu'
gen = UNetGenerator().to(device)
load_checkpoint('checkpoints/gen_latest.pth', gen)
gen.eval()

# Load a few val images and show before/after
dataset = ColorizationDataset(DATA_DIR, split='val', image_size=IMAGE_SIZE)
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
for i in range(3):
    L, ab_real = dataset[i]
    with torch.no_grad():
        ab_fake = gen(L.unsqueeze(0).to(device)).cpu().squeeze(0)
    gray = (L.squeeze().numpy() + 1) / 2
    rgb_real = lab_to_rgb(L, ab_real)
    rgb_fake = lab_to_rgb(L, ab_fake)
    axes[i][0].imshow(gray, cmap='gray'); axes[i][0].set_title('Grayscale')
    axes[i][1].imshow(rgb_fake); axes[i][1].set_title('Colorized')
    axes[i][2].imshow(rgb_real); axes[i][2].set_title('Original')
    for ax in axes[i]: ax.axis('off')
plt.tight_layout()
plt.savefig(f'{DRIVE_DIR}/sample_results.png', dpi=150)
plt.show()

In [ ]:
import subprocess, os
from google.colab import files
from IPython.display import Image as IPImage, display

uploaded = files.upload()
input_path = list(uploaded.keys())[0]

# Use subprocess to avoid shell splitting on spaces/unicode in filename
safe_name = os.path.basename(input_path).replace(' ', '_')
output_path = f'{DRIVE_DIR}/colorized_{safe_name}'

result = subprocess.run(
    ['python', 'inference.py',
     '--checkpoint', 'checkpoints/gen_latest.pth',
     '--input', input_path,
     '--output', output_path],
    capture_output=True, text=True
)
if result.returncode != 0:
    print("Error:\n", result.stderr)
else:
    display(IPImage(output_path))